Nama : Muhammad Khasan Zubaeri

NIM : 250401020089

Kelas : IF403

In [24]:
# Langkah 1: Generate & Eksplorasi Dataset Transaksi
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
  transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

# Menghitung frekuensi setiap produk
from collections import Counter

frekuensi = Counter()

for t in transaksi:
    frekuensi.update(t)

# Membuat DataFrame dan mengurutkan
df_frekuensi = (
    pd.DataFrame(frekuensi.items(), columns=['Produk', 'Frekuensi'])
)

df_frekuensi.insert(0, 'No', range(1, len(df_frekuensi) + 1))
print(df_frekuensi.to_string(index=False))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50
 No  Produk  Frekuensi
  1    Keju         17
  2    Roti         16
  3 Mentega         21
  4    Kopi         16
  5   Selai         26
  6     Teh         23
  7    Susu         16
  8   Telur         18
  9    Gula         16
 10  Sereal          9


In [25]:
# Langkah 2: One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [33]:
# Langkah 3: Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.10, 0.20]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f"min_support = {ms}: {len(freq)} itemset ditemukan")

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.10, use_colnames=True)
freq_items = freq_items.sort_values(by='support', ascending=False)

print(freq_items.head(10))

min_support = 0.05: 74 itemset ditemukan
min_support = 0.1: 44 itemset ditemukan
min_support = 0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [34]:
# Langkah 4: Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                          min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))

         antecedents consequents  support  confidence      lift
10       (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
13  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
15   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Aturan mana yang paling kuat (Lift tertinggi)?

> Aturan asosiasi yang paling kuat adalah **(Keju, Teh) - (Telur)** dengan nilai **Lift sebesar 2,38**.
Aturan ini memiliki support 0,12, yang berarti 12% dari seluruh transaksi mengandung ketiga produk tersebut. Nilai confidence sebesar 85,71% menunjukkan bahwa sebagian besar pelanggan yang membeli keju dan teh juga membeli telur.


Apakah masuk akal secara bisnis (Keju, Teh -> Telur)?

> Ya, aturan tersebut masih masuk akal secara bisnis. Produk keju dan telur sering digunakan sebagai bahan makanan atau menu sarapan, sedangkan teh merupakan minuman yang umum dikonsumsi bersamaan saat sarapan. Oleh karena itu, pelanggan yang membeli keju dan teh cenderung juga membeli telur.



In [35]:
# Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
    'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [36]:
# Langkah 6: Bandingkan Kedua Pendekatan
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Apakah kedua pendekatan memberi rekomendasi yang konsisten?


> Ya cukup konsisten karena kedua metode sama-sama merekomendasikan Selai, meskipun Content-Based Filtering dapat memberikan rekomendasi tambahan berdasarkan kemiripan kategori produk seperti Sereal dan Susu.


Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?


> **Association Rules**, sebaiknya digunakan ketika tersedia banyak data transaksi dan ingin mengetahui produk yang sering dibeli secara bersamaan.

> **Content-Based Filtering** sebaiknya digunakan ketika ingin merekomendasikan produk yang memiliki karakteristik serupa atau ketika data transaksi masih terbatas.

> **Hybrid Recommendation** digunakan ketika ingin menghasilkan rekomendasi yang lebih akurat dan lengkap.




